# ✏️ TOONIFY — Sketch LoRA Training with kohya_ss

Train a custom **Stable Diffusion 1.5 LoRA** on your 400 sketch images using **kohya_ss**.

| Setting | Value |
|---------|-------|
| Base Model | `runwayml/stable-diffusion-v1-5` |
| Resolution | 512×512 |
| LoRA Rank | 16 |
| Trigger Word | `abhi_sketch_style` |
| Epochs | 20 |
| Output | `sketch_lora.safetensors` |

> **Runtime → Change runtime type → T4 GPU** before running!

## Step 1: Install kohya_ss

In [ ]:
import os

# Clone kohya_ss with submodules
if not os.path.exists('/content/kohya_ss'):
    !git clone --recursive https://github.com/bmaltais/kohya_ss.git /content/kohya_ss

%cd /content/kohya_ss
!git submodule update --init --recursive

# Install requirements
!pip install -q --upgrade pip
!pip install -q -r requirements.txt
!pip install -q bitsandbytes==0.41.3 --no-deps

print("✅ kohya_ss installed!")

## Step 2: Download SD 1.5 Base Model

In [ ]:
import os

model_dir = '/content/models'
os.makedirs(model_dir, exist_ok=True)
model_path = f'{model_dir}/v1-5-pruned-emaonly.safetensors'

if not os.path.exists(model_path):
    print("Downloading SD 1.5 base model (~4GB)... this takes ~5 minutes")
    !wget -q --show-progress -O {model_path} \
        "https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors"
    print("✅ SD 1.5 base model downloaded!")
else:
    print(f"✅ SD 1.5 model already exists at {model_path}")

## Step 3: Mount Google Drive & Prepare Dataset

In [ ]:
import os
import zipfile
import shutil
from google.colab import drive

if os.path.exists('/content/drive'):
    print('Google Drive already mounted.')
else:
    drive.mount('/content/drive')

TRIGGER_WORD = 'abhi_sketch_style'
REPEATS = 20
kohya_dataset_dir = f'/content/train_data/{REPEATS}_{TRIGGER_WORD}'
os.makedirs(kohya_dataset_dir, exist_ok=True)

drive_folder = '/content/drive/MyDrive/sketch_dataset'
zip_path_drive = '/content/drive/MyDrive/sketch_dataset.zip'
zip_path_local = '/content/sketch_dataset.zip'

temp_dir = '/content/raw_dataset'
os.makedirs(temp_dir, exist_ok=True)

if os.path.exists(drive_folder):
    print('✅ Found sketch_dataset folder in Google Drive!')
    for root, dirs, files in os.walk(drive_folder):
        for file in files:
            if file.endswith(('.png', '.jpg', '.jpeg', '.txt')):
                shutil.copy2(os.path.join(root, file), os.path.join(temp_dir, file))
elif os.path.exists(zip_path_drive) or os.path.exists(zip_path_local):
    zip_target = zip_path_drive if os.path.exists(zip_path_drive) else zip_path_local
    with zipfile.ZipFile(zip_target, 'r') as z:
        z.extractall(temp_dir)

copied = 0
for root, dirs, files in os.walk(temp_dir):
    for file in files:
        if file.endswith(('.png', '.jpg', '.jpeg', '.txt')):
            shutil.copy2(os.path.join(root, file), os.path.join(kohya_dataset_dir, file))
            copied += 1

print(f'✅ Copied {copied} files into kohya_ss dataset folder: {kohya_dataset_dir}')

## Step 4: Verify Dataset

In [ ]:
import glob

images = glob.glob(f'{kohya_dataset_dir}/*.png') + glob.glob(f'{kohya_dataset_dir}/*.jpg')
captions = glob.glob(f'{kohya_dataset_dir}/*.txt')

print(f'✅ Images : {len(images)}')
print(f'✅ Captions: {len(captions)}')

if images and captions:
    with open(captions[0], 'r') as f:
        print(f'\nSample caption: {f.read().strip()}')

## Step 5: Train Sketch LoRA with kohya_ss

In [ ]:
import os

OUTPUT_DIR = '/content/output_lora'
LOG_DIR = '/content/logs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

MODEL_PATH = '/content/models/v1-5-pruned-emaonly.safetensors'
DATASET_DIR = '/content/train_data'
TRIGGER_WORD = 'abhi_sketch_style'

config_dir = '/content/kohya_config'
os.makedirs(config_dir, exist_ok=True)

dataset_config = f"""
[general]
shuffle_caption = true
caption_extension = ".txt"
keep_tokens = 1

[[datasets]]
resolution = 512
batch_size = 2

  [[datasets.subsets]]
  image_dir = "{DATASET_DIR}/20_{TRIGGER_WORD}"
  class_tokens = "{TRIGGER_WORD}"
  num_repeats = 20
"""

with open(f'{config_dir}/dataset.toml', 'w') as f:
    f.write(dataset_config)

print("✅ Dataset config written.")
print("🚀 Starting LoRA training... (this will take 20-40 minutes on T4)")

!python /content/kohya_ss/sd-scripts/train_network.py \
    --pretrained_model_name_or_path="{MODEL_PATH}" \
    --dataset_config="{config_dir}/dataset.toml" \
    --output_dir="{OUTPUT_DIR}" \
    --output_name="sketch_lora" \
    --logging_dir="{LOG_DIR}" \
    --save_model_as=safetensors \
    --network_module=networks.lora \
    --network_dim=16 \
    --network_alpha=16 \
    --learning_rate=1e-4 \
    --unet_lr=1e-4 \
    --text_encoder_lr=5e-5 \
    --lr_scheduler=cosine \
    --lr_warmup_steps=100 \
    --max_train_epochs=20 \
    --train_batch_size=2 \
    --gradient_accumulation_steps=4 \
    --mixed_precision=fp16 \
    --save_precision=fp16 \
    --optimizer_type=AdamW8bit \
    --xformers \
    --cache_latents \
    --save_every_n_epochs=5 \
    --clip_skip=2 \
    --seed=42

print("\n✅ Training complete!")

## Step 6: Save LoRA to Google Drive

In [ ]:
import shutil
import os

gdrive_output = '/content/drive/MyDrive/models'
os.makedirs(gdrive_output, exist_ok=True)

lora_file = '/content/output_lora/sketch_lora.safetensors'

if os.path.exists(lora_file):
    dest = os.path.join(gdrive_output, 'sketch_lora.safetensors')
    shutil.copy2(lora_file, dest)
    print(f'✅ Saved sketch_lora.safetensors to Google Drive: {dest}')
else:
    import glob
    files = glob.glob('/content/output_lora/*.safetensors')
    if files:
        latest = sorted(files)[-1]
        dest = os.path.join(gdrive_output, os.path.basename(latest))
        shutil.copy2(latest, dest)
        print(f'✅ Saved {os.path.basename(latest)} to Google Drive!')
    else:
        print('⚠️ No .safetensors found — check training logs above.')